---

# 🎓 Self-Assignment: Build Your Own RAG System

## Mini Project — "Ask My Documents"

**Objective:** Build a complete, end-to-end RAG application using everything you learned in this course. You will create a system that can answer questions about **your own documents** (PDFs, text files, or web pages).

**Estimated Time:** 2–3 hours

**Difficulty:** ⭐⭐⭐ Intermediate

---

### 📋 Project Brief

> *You are an AI engineer at a company. Your team has a collection of internal documents (policies, technical guides, meeting notes). Build a RAG-powered Q&A system that lets employees ask natural language questions and get accurate, grounded answers from these documents.*

---

### ✅ Requirements

Your submission must include a working Jupyter notebook with the following **7 tasks**:

| Task | Description | Points |
|------|-------------|--------|
| **Task 1** | Load at least **3 documents** from at least **2 different sources** (e.g., PDF + Web, or PDF + TXT) | 10 |
| **Task 2** | Implement a chunking strategy with a **justified choice** of `chunk_size` and `chunk_overlap` — write a comment explaining your reasoning | 10 |
| **Task 3** | Store embeddings in a **Chroma** vector store with persistence enabled | 10 |
| **Task 4** | Build a **retriever** and demonstrate it works by showing top-K results for 3 different queries | 10 |
| **Task 5** | Write a **custom RAG prompt** that includes a system message with specific instructions (e.g., "answer in bullet points", "cite the source page", or "say I don't know if unsure") | 15 |
| **Task 6** | Assemble the **complete RAG chain** using LCEL (pipe operators) and test it with at least **5 questions** | 15 |
| **Task 7** | **Bonus Challenges** (pick at least one) — see below | 30 |

**Total: 100 points**

---

### 🌟 Task 7 — Bonus Challenges (pick at least one for full marks)

| Challenge | Description | Points |
|-----------|-------------|--------|
| **A. Multi-turn memory** | Add conversation history so follow-up questions work (e.g., "What about its pricing?" after asking about a product) | 10 |
| **B. Source citation** | Modify the chain to return **which documents** were used to answer each question (page number, filename) | 10 |
| **C. Evaluation harness** | Create 5 question-answer pairs as ground truth, then run your RAG chain and compare its answers against the ground truth (simple string match or LLM-as-judge) | 10 |
| **D. Chunking experiment** | Try 3 different chunk sizes (e.g., 200, 500, 1000) and compare retrieval quality — which size finds the best context for the same query? | 10 |
| **E. Hybrid retriever** | Combine similarity search with MMR or metadata filtering — explain when each strategy is better | 10 |
| **F. Streaming UI** | Build a simple interactive cell where users type questions and see the RAG chain stream its answer token by token | 10 |

---

### 🏗️ Starter Scaffold

The cells below provide a **skeleton structure** for your project. Each cell has `TODO` comments where you need to fill in your code. The structure follows the same 4-phase approach from the course.

> **Tip:** Refer back to the demo cells above (Cells 1–22) whenever you get stuck. The patterns are the same — you're just applying them to your own data now.

---

#### Task 1 — Load Your Documents

In [6]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install -q langchain langchain-openai langchain-chroma langchain-community langchain-text-splitters langchain-core chromadb openai python-dotenv pypdf

In [ ]:
# ============================================================
# AZURE OPENAI CONFIGURATION
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT            = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY             = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION         = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
AZURE_OPENAI_CHAT_DEPLOYMENT     = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4.1-mini")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")

print("Azure OpenAI configuration loaded")
print(f"   Endpoint   : {AZURE_OPENAI_ENDPOINT}")
print(f"   Chat model : {AZURE_OPENAI_CHAT_DEPLOYMENT}")
print(f"   Embed model: {AZURE_OPENAI_EMBEDDING_DEPLOYMENT}")

Azure OpenAI configuration loaded
   Endpoint   : https://group2-salomous-resource.services.ai.azure.com
   Chat model : gpt-4.1-mini
   Embed model: text-embedding-3-small


In [9]:
# ============================================================
# TASK 1: Load Your Documents (at least 3 docs, 2+ sources)
# ============================================================
# Domain: Mitsubishi automotive product brochures (Indonesia)
# Source 1: PDF brosur dari folder dataset
# Source 2: Teks inline produk Mitsubishi sebagai dokumen tambahan
# Source 3: Teks inline FAQ kendaraan niaga sebagai sumber ketiga

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_core.documents import Document
import glob
import os

# ── Source 1: PDF brosur Mitsubishi dari folder dataset ──
DATASET_FOLDER = "D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 31 - RAG Optimization Re-ranking, Caching & Scaling/dataset"

pdf_docs = []
if os.path.exists(DATASET_FOLDER):
    pdf_files = sorted(glob.glob(os.path.join(DATASET_FOLDER, "*.pdf")))
    for pdf_path in pdf_files:
        try:
            loader = PyPDFLoader(pdf_path)
            pages = loader.load()
            pdf_docs.extend(pages)
        except Exception as e:
            print(f"   Skipped {os.path.basename(pdf_path)}: {e}")
    print(f"PDF documents loaded: {len(pdf_docs)} pages from {len(pdf_files)} files")
else:
    print("Dataset folder not found — using inline fallback documents")

# ── Source 2: Inline product knowledge (passenger cars) ──
passenger_texts = [
    """Mitsubishi Xpander adalah MPV 7-penumpang dengan mesin 1.5L MIVEC 105 PS.
    Transmisi tersedia Manual 5-percepatan dan CVT otomatis.
    Fitur unggulan: Easy Select 4WD (Xpander Cross), Apple CarPlay, Android Auto,
    panoramic monitor, dan kursi baris kedua yang dapat dilipat rata.
    Konsumsi BBM sekitar 14-16 km/liter. GVW 2.030 kg.""",

    """Mitsubishi Pajero Sport menggunakan mesin diesel 2.4L MIVEC 181 PS dengan torsi 430 Nm.
    Sistem penggerak Super Select 4WD-II dengan mode 2H, 4H, 4HLc, 4LLc.
    Fitur: terrain mode selector (snow, gravel, mud, sand, rock), hill descent control,
    active stability control, 360-degree camera. Ground clearance 218 mm.
    Kapasitas tangki 68 liter. Tersedia dalam varian Exceed dan Dakar.""",

    """Mitsubishi Triton 2024 adalah pickup truck dengan mesin diesel 2.4L MIVEC turbo 204 PS.
    Torsi maksimal 470 Nm di 2.500 rpm. Transmisi otomatis 6-percepatan.
    Kapasitas bak belakang 1.100 liter. Payload hingga 1.000 kg.
    Teknologi terbaru: e-ATSS (electronic Active Traction Support System),
    hill start assist, trailer stability assist. Tersedia double cab dan single cab.""",

    """Mitsubishi Xforce adalah SUV compact dengan mesin 1.5L MIVEC 105 PS.
    Desain sporty dengan ground clearance 222 mm, tertinggi di kelasnya.
    Fitur: PHEV ready platform, 360-degree camera, wireless charging,
    panoramic sunroof, 8 speaker premium audio.
    Kapasitas bagasi 564 liter dengan kursi belakang tegak.""",
]

web_docs = [
    Document(
        page_content=text,
        metadata={"source": "mitsubishi_passenger_knowledge", "domain": "passenger", "page": i + 1}
    )
    for i, text in enumerate(passenger_texts)
]

# ── Source 3: Inline FAQ kendaraan niaga ──
niaga_texts = [
    """Mitsubishi Colt Diesel FE 71 adalah truk ringan dengan mesin 4D34-2AT5 Turbo Intercooler.
    Spesifikasi mesin: 4 silinder, 3.908 cc, daya 110 PS di 2.900 rpm, torsi 28 kgm di 1.600 rpm.
    GVW 5.150 kg, berat chassis 1.835 kg. Transmisi M025S5 5 maju 1 mundur.
    Kapasitas tangki 70 liter. Jarak sumbu roda 2.500 mm.
    Cocok untuk distribusi barang skala menengah di perkotaan.""",

    """Mitsubishi Colt Diesel FE 74 HD adalah truk medium bertenaga tinggi.
    Mesin 4D34-2AT5 dengan daya 136 PS. GVW 8.500 kg.
    Tersedia varian HD (heavy duty) dan HDS untuk operasi berat.
    Cocok untuk konstruksi, tambang, dan distribusi antar kota.
    Sistem pengereman: drum brake hydraulic dengan vacuum servo.""",

    """Mitsubishi eCanter adalah truk listrik full-electric untuk kendaraan niaga perkotaan.
    Jarak tempuh hingga 120 km per pengisian penuh.
    Kapasitas muatan 2.750 kg. Motor listrik 129 kW / 390 Nm.
    Pengisian menggunakan CHAdeMO DC fast charging (selesai dalam 1,5 jam).
    Zero emisi — sangat cocok untuk distribusi di kawasan zero-emission zone.""",

    """Mitsubishi Fuso Destinator adalah truk besar untuk angkutan jarak jauh.
    Mesin diesel 6 silinder 7.545 cc, daya 260 PS.
    GVW 26.000 kg. Transmisi otomatis 12-percepatan.
    Fitur: lane departure warning, adaptive cruise control, forward collision warning.
    Radius putar minimum 9,4 meter. Kapasitas tangki 2x 200 liter.""",
]

txt_docs = [
    Document(
        page_content=text,
        metadata={"source": "mitsubishi_niaga_faq", "domain": "commercial", "page": i + 1}
    )
    for i, text in enumerate(niaga_texts)
]

# ── Gabungkan semua dokumen ──
all_docs = pdf_docs + web_docs + txt_docs

print(f"\nSumber dokumen:")
print(f"   PDF brosur  : {len(pdf_docs)} halaman")
print(f"   Passenger KB: {len(web_docs)} dokumen")
print(f"   Niaga FAQ   : {len(txt_docs)} dokumen")
print(f"   Total       : {len(all_docs)} dokumen")

PDF documents loaded: 71 pages from 19 files

Sumber dokumen:
   PDF brosur  : 71 halaman
   Passenger KB: 4 dokumen
   Niaga FAQ   : 4 dokumen
   Total       : 79 dokumen


#### Task 2 — Chunk Your Documents

Choose your `chunk_size` and `chunk_overlap` and **explain why** in a comment.

In [10]:
# ============================================================
# TASK 2: Chunking Strategy
# ============================================================
# Justifikasi pemilihan chunk_size dan chunk_overlap:
#
# chunk_size = 800:
#   Brosur Mitsubishi berisi tabel spesifikasi padat dan deskripsi fitur
#   yang saling terkait. Chunk terlalu kecil (200-300) akan memotong
#   tabel spesifikasi di tengah sehingga data menjadi tidak lengkap.
#   Chunk terlalu besar (1500+) memasukkan terlalu banyak konteks tidak
#   relevan ke dalam satu retrieval. Ukuran 800 cukup untuk menampung
#   satu section spesifikasi lengkap (misal: seluruh bagian "Mesin").
#
# chunk_overlap = 150:
#   Overlap 150 karakter memastikan kalimat yang berada di batas chunk
#   tidak kehilangan konteksnya. Untuk dokumen teknis, angka seperti
#   "110 PS" atau "GVW 5.150 kg" harus selalu muncul bersama unit
#   dan labelnya — overlap mencegah pemotongan di tengah informasi kritis.

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(all_docs)

print(f"Dokumen asli   : {len(all_docs)}")
print(f"Chunks dibuat  : {len(chunks)}")
print(f"Rata-rata panjang: {sum(len(c.page_content) for c in chunks) // len(chunks)} karakter")
print(f"\nContoh chunk pertama:")
print(chunks[0].page_content[:300])
print(f"Metadata: {chunks[0].metadata}")

Dokumen asli   : 79
Chunks dibuat  : 91
Rata-rata panjang: 607 karakter

Contoh chunk pertama:
SEPTEMBER 23
www.ktbfuso.co.id
FE 71L I 108PS I 4 Ban
Metadata: {'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 27.8 (Macintosh)', 'creationdate': '2023-12-27T14:56:13+07:00', 'gts_pdfxversion': 'PDF/X-4', 'moddate': '2023-12-27T14:56:13+07:00', 'title': 'Flyer FE 71 Long - Sep 23-WEB', 'trapped': '/False', 'source': 'D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 31 - RAG Optimization Re-ranking, Caching & Scaling/dataset\\mitsubishi-colt-fe-71-l-240729.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


#### Task 3 — Store in ChromaDB

In [11]:
# ============================================================
# TASK 3: Create Chroma Vector Store
# ============================================================
from langchain_chroma import Chroma
from langchain_openai import AzureOpenAIEmbeddings
import shutil, os

# Inisialisasi Azure embeddings
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Buat Chroma vector store dengan persistence ke disk
PERSIST_DIR = "./mitsubishi_chroma_db1"
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="mitsubishi_product_kb",
)

print(f"Tersimpan {len(chunks)} chunks ke Chroma")
print(f"   Collection : mitsubishi_product_kb")
print(f"   Persisted  : {PERSIST_DIR}/")

Tersimpan 91 chunks ke Chroma
   Collection : mitsubishi_product_kb
   Persisted  : ./mitsubishi_chroma_db1/


#### Task 4 — Build & Test the Retriever

In [12]:
# ============================================================
# TASK 4: Retriever — Test with 3 Different Queries
# ============================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

test_queries = [
    "Apa spesifikasi mesin Mitsubishi Colt Diesel FE 71?",
    "Kendaraan apa yang cocok untuk distribusi barang antar kota kapasitas besar?",
    "Apa perbedaan Xpander dan Xpander Cross?",
]

for query in test_queries:
    docs = retriever.invoke(query)
    print(f"\nQuery: \"{query}\"")
    print(f"   Results: {len(docs)} chunks retrieved")
    for i, doc in enumerate(docs):
        print(f"   [{i+1}] \"{doc.page_content[:100]}...\"")
        print(f"         metadata: {doc.metadata}")


Query: "Apa spesifikasi mesin Mitsubishi Colt Diesel FE 71?"
   Results: 4 chunks retrieved
   [1] "Mitsubishi Colt Diesel FE 71 adalah truk ringan dengan mesin 4D34-2AT5 Turbo Intercooler.
    Spesif..."
         metadata: {'domain': 'commercial', 'page': 1, 'source': 'mitsubishi_niaga_faq'}
   [2] "Mitsubishi Colt Diesel FE 74 HD adalah truk medium bertenaga tinggi.
    Mesin 4D34-2AT5 dengan daya..."
         metadata: {'domain': 'commercial', 'source': 'mitsubishi_niaga_faq', 'page': 2}
   [3] "ditingkatkan menjadi 6,666 
cocok untuk medan berat.
Leaf Spring Depan & Belakang
Heavy Duty, cocok ..."
         metadata: {'source': 'D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 31 - RAG Optimization Re-ranking, Caching & Scaling/dataset\\mitsubishi-colt-fe-shdx-477280.pdf', 'producer': 'Adobe PDF library 17.00', 'total_pages': 2, 'trapped': '/False', 'creator': 'Adobe Illustrator 27.8 (Macintosh)', 'page_label': '2', 'gts_pdfxversion': 'PDF

#### Task 5 — Design Your Custom RAG Prompt

Create a **custom system message** that shapes how the model answers. Be creative and specific.

In [13]:
# ============================================================
# TASK 5: Custom RAG Prompt
# ============================================================
from langchain_core.prompts import ChatPromptTemplate

# Persona: RAIKYO — AI Sales Consultant profesional Mitsubishi Indonesia
# Format  : Jawaban dalam poin-poin, selalu sebutkan nama model produk,
#           sertakan spesifikasi teknis jika relevan.
# Fallback: Jika konteks tidak cukup, katakan secara jujur.
# Bahasa  : Indonesia, profesional namun ramah.

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Kamu adalah RAIKYO, AI Sales Consultant profesional untuk produk Mitsubishi Indonesia. "
     "Tugasmu adalah membantu pelanggan memilih kendaraan yang paling sesuai dengan kebutuhan mereka. "
     "Panduan menjawab:\n"
     "- Jawab menggunakan HANYA informasi dari konteks yang diberikan.\n"
     "- Gunakan format poin-poin agar mudah dibaca.\n"
     "- Selalu sebutkan nama model produk secara spesifik.\n"
     "- Sertakan data teknis (mesin, kapasitas, daya) jika relevan dengan pertanyaan.\n"
     "- Jika konteks tidak mengandung informasi yang cukup, katakan: "
     "'Informasi tersebut tidak tersedia dalam dokumen yang saya miliki.'\n"
     "- Jangan mengarang fakta atau spesifikasi di luar konteks."),
    ("human",
     "Konteks dari brosur Mitsubishi:\n{context}\n\nPertanyaan: {question}")
])

# Test render prompt dengan data sampel
rendered = rag_prompt.invoke({
    "context": "Mitsubishi Colt Diesel FE 71: mesin 4D34, 110 PS, GVW 5.150 kg, tangki 70 liter.",
    "question": "Berapa kapasitas tangki FE 71?"
})

for msg in rendered.messages:
    print(f"[{msg.type.upper()}] {msg.content[:300]}...")

[SYSTEM] Kamu adalah RAIKYO, AI Sales Consultant profesional untuk produk Mitsubishi Indonesia. Tugasmu adalah membantu pelanggan memilih kendaraan yang paling sesuai dengan kebutuhan mereka. Panduan menjawab:
- Jawab menggunakan HANYA informasi dari konteks yang diberikan.
- Gunakan format poin-poin agar mu...
[HUMAN] Konteks dari brosur Mitsubishi:
Mitsubishi Colt Diesel FE 71: mesin 4D34, 110 PS, GVW 5.150 kg, tangki 70 liter.

Pertanyaan: Berapa kapasitas tangki FE 71?...


#### Task 6 — Assemble & Test the Full RAG Chain

In [18]:
# ============================================================
# TASK 6: Complete RAG Chain — Test with 5+ Questions
# ============================================================
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# DEBUGGING: Print deployment name yang akan digunakan
print(f"Using Azure deployment: {AZURE_OPENAI_CHAT_DEPLOYMENT}")
print(f"Using endpoint: {AZURE_OPENAI_ENDPOINT}")

# Inisialisasi model Azure dengan deployment yang benar
model = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
    max_tokens=1000,
)

# PASTIKAN retriever sudah dibuat dari Task 4
# Jika Task 4 sudah dijalankan, retriever sudah ada di memory
# Jika belum, buat ulang di sini:
try:
    # Cek apakah retriever sudah ada
    _ = retriever
    print("Retriever sudah tersedia dari Task 4")
except NameError:
    # Jika belum, buat retriever baru
    print("Retriever belum ada, membuat ulang...")
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )
    print("Retriever berhasil dibuat")

# Helper untuk format dokumen yang diambil
def format_docs(docs):
    """Format retrieved documents menjadi string context."""
    parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        # Ambil nama file saja dari path panjang
        if "/" in source or "\\" in source:
            source = source.split("/")[-1].split("\\")[-1]
        page = doc.metadata.get("page", "?")
        content = doc.page_content[:200].replace("\n", " ")
        parts.append(f"[Dok {i+1}] {source} hal.{page}: {content}...")
    return "\n\n".join(parts)

# Assemble RAG chain menggunakan LCEL
rag_chain = (
    RunnableParallel(
        context  = retriever | format_docs,
        question = RunnablePassthrough()
    )
    | rag_prompt
    | model
    | StrOutputParser()
)

print("RAG chain berhasil dibuat")

# Test dengan 5 pertanyaan
questions = [
    "Apa spesifikasi mesin dan daya Mitsubishi Colt Diesel FE 71?",
    "Berapa kapasitas tangki bahan bakar FE 71 dan FE 74?",
    "Kendaraan apa yang paling cocok untuk distribusi barang di kawasan zero-emission?",
    "Bandingkan ground clearance antara Xpander Cross dan Xforce",
    "Mitsubishi punya kendaraan elektrik tidak? Jelaskan spesifikasinya.",
]

print("\n" + "=" * 60)
print("  🧬 MITSUBISHI RAG SYSTEM — TEST RESULTS")
print("=" * 60)

for i, q in enumerate(questions, 1):
    print(f"\n[{i}/5] Processing query...")
    try:
        answer = rag_chain.invoke(q)
        print(f" {q}")
        print(f" {answer}")
        print("-" * 50)
    except Exception as e:
        print(f" Error pada query {i}: {str(e)}")
        print(f"   Query: {q}")
        print("-" * 50)
        # Lanjutkan ke query berikutnya
        continue

print("\n✅ Testing selesai!")

Using Azure deployment: gpt-4.1-mini
Using endpoint: https://group2-salomous-resource.services.ai.azure.com
Retriever sudah tersedia dari Task 4
RAG chain berhasil dibuat

  🧬 MITSUBISHI RAG SYSTEM — TEST RESULTS

[1/5] Processing query...
 Apa spesifikasi mesin dan daya Mitsubishi Colt Diesel FE 71?
 Berikut spesifikasi mesin dan daya Mitsubishi Colt Diesel FE 71 berdasarkan dokumen yang tersedia:

- Mesin: 4D34-2AT5 Turbo Intercooler
- Konfigurasi: 4 silinder
- Kapasitas mesin: 3.908 cc
- Daya maksimum: 110 PS pada 2.900 rpm
- Torsi maksimum: 28 kgm pada 1.600 rpm
--------------------------------------------------

[2/5] Processing query...
 Berapa kapasitas tangki bahan bakar FE 71 dan FE 74?
 Berikut kapasitas tangki bahan bakar untuk model Mitsubishi yang Anda tanyakan:

- Mitsubishi FE 71 (FE 71L dan FE SHDX): kapasitas tangki bahan bakar 100 liter (informasi dari Dok 1 dan Dok 4)
- Mitsubishi FE 74: Informasi kapasitas tangki bahan bakar tidak tersedia dalam dokumen yang saya mi

#### Task 7 — Bonus Challenge(s)

Pick **at least one** bonus challenge from the table above and implement it below.

In [19]:
# ============================================================
# TASK 7: Bonus Challenge
# Implementasi: Challenge B (Source Citation) +
#               Challenge E (Hybrid Retriever dengan MMR)
# ============================================================

# ── Challenge B: Source Citation ──
# Modifikasi chain untuk mengembalikan dokumen sumber yang digunakan

from langchain_core.runnables import RunnableLambda

def format_docs_with_citation(docs):
    """Format dokumen dengan metadata sumber lengkap untuk sitasi."""
    parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "-")
        domain = doc.metadata.get("domain", "-")
        parts.append(
            f"[Source {i+1} | File: {source} | Page: {page} | Domain: {domain}]\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(parts)

def get_sources(docs):
    """Ekstrak list sumber dokumen."""
    sources = []
    for i, doc in enumerate(docs):
        sources.append({
            "source_id": i + 1,
            "file": doc.metadata.get("source", "unknown"),
            "page": doc.metadata.get("page", "-"),
            "domain": doc.metadata.get("domain", "-"),
            "snippet": doc.page_content[:80] + "..."
        })
    return sources

# RAG chain dengan source citation
citation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Kamu adalah RAIKYO, AI Sales Consultant Mitsubishi Indonesia. "
     "Jawab menggunakan HANYA informasi dari konteks. "
     "Setiap fakta WAJIB diikuti dengan sitasi [Source N]. "
     "Contoh: 'Daya mesin 110 PS [Source 1], kapasitas tangki 70 liter [Source 2].' "
     "Jika tidak ada informasi yang cukup, katakan secara jujur."),
    ("human",
     "Konteks:\n{context}\n\nPertanyaan: {question}")
])

retrieved_docs_store = []

def retrieve_and_store(query):
    """Ambil dokumen dan simpan untuk ditampilkan sebagai sumber."""
    docs = retriever.invoke(query)
    retrieved_docs_store.clear()
    retrieved_docs_store.extend(docs)
    return docs

citation_chain = (
    RunnableParallel(
        context=RunnableLambda(retrieve_and_store) | format_docs_with_citation,
        question=RunnablePassthrough()
    )
    | citation_prompt
    | model
    | StrOutputParser()
)

# Test Citation Chain
print("=" * 60)
print("  CHALLENGE B: SOURCE CITATION")
print("=" * 60)

test_q = "Apa spesifikasi mesin dan kapasitas angkut Mitsubishi Colt Diesel FE 71?"
answer_with_citation = citation_chain.invoke(test_q)

print(f"\nPertanyaan: {test_q}")
print(f"\nJawaban dengan Sitasi:\n{answer_with_citation}")
print(f"\nDokumen yang digunakan:")
for src in get_sources(retrieved_docs_store):
    print(f"   [Source {src['source_id']}] {src['file']} | hal. {src['page']}")
    print(f"             '{src['snippet']}'")

  CHALLENGE B: SOURCE CITATION

Pertanyaan: Apa spesifikasi mesin dan kapasitas angkut Mitsubishi Colt Diesel FE 71?

Jawaban dengan Sitasi:
Spesifikasi mesin Mitsubishi Colt Diesel FE 71 adalah mesin 4 silinder dengan kapasitas 3.908 cc, menggunakan tipe 4D34-2AT5 Turbo Intercooler, menghasilkan daya 110 PS pada 2.900 rpm dan torsi 28 kgm pada 1.600 rpm. Kapasitas tangki bahan bakar adalah 70 liter. Transmisi menggunakan M025S5 dengan 5 percepatan maju dan 1 mundur. Berat chassis adalah 1.835 kg dengan GVW (Gross Vehicle Weight) sebesar 5.150 kg. Jarak sumbu roda 2.500 mm. Truk ini cocok untuk distribusi barang skala menengah di perkotaan [Source 1].

Dokumen yang digunakan:
   [Source 1] mitsubishi_niaga_faq | hal. 1
             'Mitsubishi Colt Diesel FE 71 adalah truk ringan dengan mesin 4D34-2AT5 Turbo Int...'
   [Source 2] mitsubishi_niaga_faq | hal. 2
             'Mitsubishi Colt Diesel FE 74 HD adalah truk medium bertenaga tinggi.
    Mesin 4...'
   [Source 3] D:/Users/bsi802

In [20]:
# ── Challenge E: Hybrid Retriever (Similarity vs MMR) ──
# MMR (Maximal Marginal Relevance) memilih dokumen yang relevan
# sekaligus beragam — menghindari retrieval dokumen duplikat yang
# membahas hal sama berulang-ulang.
#
# Kapan similarity lebih baik:
#   - Pertanyaan spesifik yang butuh satu jawaban tepat
#   - Contoh: "Berapa daya mesin FE 71?" → cukup 1 dokumen yang tepat
#
# Kapan MMR lebih baik:
#   - Pertanyaan perbandingan atau rekomendasi yang butuh beragam konteks
#   - Contoh: "Rekomendasikan kendaraan niaga" → butuh info dari berbagai model

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 10, "lambda_mult": 0.5}
)

similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

comparison_query = "Rekomendasikan kendaraan Mitsubishi untuk usaha angkutan barang"

sim_docs = similarity_retriever.invoke(comparison_query)
mmr_docs = mmr_retriever.invoke(comparison_query)

print("=" * 60)
print("  CHALLENGE E: SIMILARITY vs MMR RETRIEVAL")
print("=" * 60)
print(f"\nQuery: \"{comparison_query}\"")

print(f"\nSimilarity Search ({len(sim_docs)} docs):")
for i, d in enumerate(sim_docs):
    print(f"   [{i+1}] {d.metadata.get('source','-')} | '{d.page_content[:80]}...'")

print(f"\nMMR Search ({len(mmr_docs)} docs):")
for i, d in enumerate(mmr_docs):
    print(f"   [{i+1}] {d.metadata.get('source','-')} | '{d.page_content[:80]}...'")

# Cek keberagaman sumber
sim_sources = set(d.metadata.get('source', '-') for d in sim_docs)
mmr_sources = set(d.metadata.get('source', '-') for d in mmr_docs)
print(f"\nJumlah sumber unik — Similarity: {len(sim_sources)}, MMR: {len(mmr_sources)}")
print("MMR memberikan keberagaman lebih tinggi untuk pertanyaan rekomendasi.")

  CHALLENGE E: SIMILARITY vs MMR RETRIEVAL

Query: "Rekomendasikan kendaraan Mitsubishi untuk usaha angkutan barang"

Similarity Search (4 docs):
   [1] D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 31 - RAG Optimization Re-ranking, Caching & Scaling/dataset\mitsubishi-ecanter-243699.pdf | 'kondisi terbaik. Bahkan suku cadang terkecil pun dirancang untuk memenuhi kebutu...'
   [2] mitsubishi_niaga_faq | 'Mitsubishi Fuso Destinator adalah truk besar untuk angkutan jarak jauh.
    Mesi...'
   [3] D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 31 - RAG Optimization Re-ranking, Caching & Scaling/dataset\mitsubishi-colt-fe-shdx-477280.pdf | 'ditingkatkan menjadi 6,666 
cocok untuk medan berat.
Leaf Spring Depan & Belakan...'
   [4] D:/Users/bsi80274/OneDrive - PT. Berlian Sistem Informasi/AI-BOOTCAMP/TRAINING/hactiv8/sesi 31 - RAG Optimization Re-ranking, Caching & Scaling/dataset\mitsubishi-fe-84gs-6

---

### 📦 Submission Checklist

Before submitting, verify the following:

- [x] **Task 1:** Loaded 3+ documents from 2+ different source types
- [x] **Task 2:** Chunking strategy implemented with a written justification comment
- [x] **Task 3:** Chroma vector store created with persistence to disk
- [x] **Task 4:** Retriever tested with 3 queries, showing top-K results with metadata
- [x] **Task 5:** Custom RAG prompt with a specific, thoughtful system message
- [x] **Task 6:** Full RAG chain assembled with LCEL pipes, tested with 5+ questions
- [x] **Task 7:** Bonus B (Source Citation) + Bonus E (Hybrid MMR) completed
- [x] **All cells run** top-to-bottom without errors (Kernel → Restart & Run All)
- [x] **No hardcoded API keys** — uses environment variables or `.env` file

### 📁 What to Submit

1. This notebook (`.ipynb`) with all cells executed and outputs visible
2. Your `.env.example` file (with placeholder values, NOT real keys)
3. A short `README.md` (3–5 sentences) describing:
   - What documents you chose and why
   - What bonus challenge(s) you completed
   - One thing you learned or found surprising

---

### 💡 Tips for Success

- **Start with small, simple documents** — don't load 500 pages on your first try.
- **Test each task independently** before chaining them together.
- **Print intermediate results** — check what your retriever returns before building the full chain.
- **Experiment with chunk sizes** — this is the single biggest lever for RAG quality.
- **Read the error messages** — LangChain errors are usually descriptive and tell you exactly what's wrong.

Good luck! 🚀

---

## 🧹 Cleanup

In [ ]:
# ============================================================
# CLEANUP: Remove persisted Chroma database
# ============================================================
import shutil, os

if os.path.exists("./mitsubishi_chroma_db"):
    shutil.rmtree("./mitsubishi_chroma_db")
    print("Removed './mitsubishi_chroma_db/' directory")
else:
    print("Nothing to clean up")

print("\nDemo complete! You've built a full RAG pipeline with Azure OpenAI + ChromaDB + LangChain.")